# EcoGraphRAG — Notebook 2: Flat RAG Baseline

**Goal:** Run 500 HotpotQA questions through Flat RAG (FAISS + Mistral-7B 4-bit) and record baseline results.

**Hardware:** Colab T4 GPU (12 GB VRAM)

**Inputs:** `faiss_index.bin`, `chunk_mapping.json`, `embeddings.npy`, `hotpotqa_500.json`

**Outputs:** `flat_rag_results.json` with predictions, gold answers, latency, GPU memory

## 1. Setup & Drive Mount

In [ ]:
from google.colab import drive
import os, json, time

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/graphrag_research/'
DRIVE_DATA = DRIVE_ROOT + 'data/'
DRIVE_INDICES = DRIVE_ROOT + 'indices/'
DRIVE_OUTPUTS = DRIVE_ROOT + 'outputs/'
DRIVE_CHECKPOINTS = DRIVE_ROOT + 'checkpoints/'

for d in [DRIVE_DATA, DRIVE_INDICES, DRIVE_OUTPUTS, DRIVE_CHECKPOINTS]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu

In [ ]:
import torch
import numpy as np

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 2. Load Data & Index

In [ ]:
import faiss

# Load dataset
HOTPOTQA_FILE = DRIVE_DATA + 'hotpotqa_500.json'
with open(HOTPOTQA_FILE, 'r') as f:
    dataset = json.load(f)
questions = dataset['questions']
print(f'Loaded {len(questions)} questions')

# Load chunks
CHUNKS_FILE = DRIVE_DATA + 'chunks.json'
with open(CHUNKS_FILE, 'r') as f:
    chunks = json.load(f)
chunk_lookup = {c['chunk_id']: c for c in chunks}
print(f'Loaded {len(chunks)} chunks')

# Load FAISS index
FAISS_INDEX_FILE = DRIVE_INDICES + 'faiss_index.bin'
index = faiss.read_index(FAISS_INDEX_FILE)
print(f'FAISS index: {index.ntotal} vectors')

# Load chunk mapping
CHUNK_MAPPING_FILE = DRIVE_INDICES + 'chunk_mapping.json'
with open(CHUNK_MAPPING_FILE, 'r') as f:
    chunk_mapping = json.load(f)
print(f'Chunk mapping: {len(chunk_mapping)} entries')

# Load embeddings (for query encoding reference)
EMBEDDINGS_FILE = DRIVE_INDICES + 'embeddings.npy'
embeddings = np.load(EMBEDDINGS_FILE)
print(f'Embeddings: {embeddings.shape}')

## 3. Load Embedding Model (for queries)

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = 'nomic-ai/nomic-embed-text-v1'
FAISS_TOP_K = 5

embed_model = SentenceTransformer(EMBEDDING_MODEL, trust_remote_code=True)
print(f'Embedding model loaded: {EMBEDDING_MODEL}')

def faiss_retrieve(question: str, top_k: int = FAISS_TOP_K):
    """Retrieve top-k chunks for a question using FAISS."""
    q_emb = embed_model.encode(
        ['search_query: ' + question],
        normalize_embeddings=True,
    ).astype(np.float32)

    scores, indices = index.search(q_emb, top_k)
    retrieved = []
    for score, idx in zip(scores[0], indices[0]):
        cid = chunk_mapping[idx]
        chunk = chunk_lookup[cid]
        retrieved.append({
            'chunk_id': cid,
            'score': float(score),
            'text': chunk['text'],
            'title': chunk['title'],
        })
    return retrieved

## 4. Load Mistral-7B (4-bit Quantized)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_MODEL = 'mistralai/Mistral-7B-Instruct-v0.2'
LLM_MAX_NEW_TOKENS = 64

print(f'Loading {LLM_MODEL} with 4-bit quantization...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.float16,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

gpu_after_load = torch.cuda.max_memory_allocated() / (1024**2)
print(f'Model loaded. GPU memory: {gpu_after_load:.0f} MB')

In [ ]:
PROMPT_TEMPLATE = """<s>[INST] Answer the following question using ONLY the provided context.
Give a short, precise answer (1-5 words). Do not explain.

Context:
{context}

Question: {question}

Answer: [/INST]"""

def generate_answer(question: str, context_chunks: list) -> str:
    """Generate an answer using Mistral-7B given retrieved context."""
    context = '\n\n'.join([c['text'] for c in context_chunks])
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=LLM_MAX_NEW_TOKENS,
            do_sample=False,
            temperature=0.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the generated tokens
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return answer

# Quick test
test_chunks = faiss_retrieve('What is the capital of France?')
test_answer = generate_answer('What is the capital of France?', test_chunks)
print(f'Test answer: {test_answer}')

## 5. Run Flat RAG Baseline (500 Questions)

Checkpoints every 25 questions to Drive. If Colab disconnects, re-run this cell to resume.

In [ ]:
import tracemalloc

CHECKPOINT_FILE = DRIVE_CHECKPOINTS + 'flat_rag_checkpoint.json'
RESULTS_FILE = DRIVE_OUTPUTS + 'flat_rag_results.json'
CHECKPOINT_INTERVAL = 25

# ── Load checkpoint if exists ──────────────────────────────
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        ckpt = json.load(f)
    results = ckpt['results']
    start_idx = ckpt['current_index'] + 1
    print(f'Resuming from checkpoint: {len(results)} results, starting at index {start_idx}')
else:
    results = []
    start_idx = 0
    print('Starting from scratch')

# ── Main experiment loop ──────────────────────────────────
total = len(questions)
print(f'Running Flat RAG on questions {start_idx} to {total - 1}...\n')

for i in range(start_idx, total):
    q = questions[i]
    question = q['question']
    gold = q['answer']
    q_type = q['type']

    # Measure cost
    torch.cuda.reset_peak_memory_stats()
    tracemalloc.start()
    t_start = time.time()

    # Retrieve
    retrieved = faiss_retrieve(question)

    # Generate
    prediction = generate_answer(question, retrieved)

    # Metrics
    latency = time.time() - t_start
    gpu_mb = torch.cuda.max_memory_allocated() / (1024**2)
    _, ram_peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    result = {
        'id': q['id'],
        'question': question,
        'prediction': prediction,
        'gold': gold,
        'type': q_type,
        'level': q.get('level', 'unknown'),
        'latency_seconds': round(latency, 3),
        'peak_gpu_mb': round(gpu_mb, 2),
        'peak_ram_mb': round(ram_peak / (1024**2), 2),
        'chunks_retrieved': len(retrieved),
        'chunk_ids': [c['chunk_id'] for c in retrieved],
    }
    results.append(result)

    # Progress
    if (i + 1) % 10 == 0 or i == total - 1:
        print(f'  [{i+1}/{total}] {latency:.2f}s  GPU:{gpu_mb:.0f}MB  '
              f'Pred:"{prediction[:40]}"  Gold:"{gold}"')

    # Checkpoint
    if (i + 1) % CHECKPOINT_INTERVAL == 0:
        ckpt_data = {'current_index': i, 'num_results': len(results), 'results': results}
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump(ckpt_data, f, indent=2, ensure_ascii=False)
        print(f'  💾 Checkpoint saved at index {i}')

# ── Save final results ────────────────────────────────────
with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f'\n✅ Saved {len(results)} results to {RESULTS_FILE}')

# Clean up checkpoint
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)
    print('Checkpoint cleared.')

## 6. Quick Evaluation

In [ ]:
import string, re
from collections import Counter

def normalize(s):
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(c for c in s if c not in string.punctuation)
    return ' '.join(s.split())

def em(pred, gold):
    return int(normalize(pred) == normalize(gold))

def f1(pred, gold):
    pt = normalize(pred).split()
    gt = normalize(gold).split()
    if not pt or not gt:
        return float(pt == gt)
    common = Counter(pt) & Counter(gt)
    nc = sum(common.values())
    if nc == 0:
        return 0.0
    p = nc / len(pt)
    r = nc / len(gt)
    return 2 * p * r / (p + r)

# Overall
em_scores = [em(r['prediction'], r['gold']) for r in results]
f1_scores = [f1(r['prediction'], r['gold']) for r in results]

print(f'\n{"="*60}')
print(f'FLAT RAG BASELINE RESULTS ({len(results)} questions)')
print(f'{"="*60}')
print(f'Overall EM:  {sum(em_scores)/len(em_scores):.4f}  ({sum(em_scores)/len(em_scores)*100:.1f}%)')
print(f'Overall F1:  {sum(f1_scores)/len(f1_scores):.4f}  ({sum(f1_scores)/len(f1_scores)*100:.1f}%)')

# By type
for qt in ['bridge', 'comparison']:
    type_results = [r for r in results if r['type'] == qt]
    if type_results:
        type_em = [em(r['prediction'], r['gold']) for r in type_results]
        type_f1 = [f1(r['prediction'], r['gold']) for r in type_results]
        print(f'\n{qt.capitalize()} ({len(type_results)} Qs):')
        print(f'  EM: {sum(type_em)/len(type_em):.4f}  F1: {sum(type_f1)/len(type_f1):.4f}')

# Latency stats
latencies = [r['latency_seconds'] for r in results]
print(f'\nLatency: avg={sum(latencies)/len(latencies):.2f}s  '
      f'median={sorted(latencies)[len(latencies)//2]:.2f}s  '
      f'max={max(latencies):.2f}s')

gpu_mbs = [r['peak_gpu_mb'] for r in results]
print(f'Peak GPU: avg={sum(gpu_mbs)/len(gpu_mbs):.0f} MB  max={max(gpu_mbs):.0f} MB')

In [ ]:
# Save metrics summary
metrics = {
    'experiment': 'flat_rag_baseline',
    'num_questions': len(results),
    'overall_em': round(sum(em_scores) / len(em_scores), 4),
    'overall_f1': round(sum(f1_scores) / len(f1_scores), 4),
    'avg_latency_s': round(sum(latencies) / len(latencies), 3),
    'max_gpu_mb': round(max(gpu_mbs), 2),
    'llm_model': LLM_MODEL,
    'quantization': '4bit_nf4',
    'embedding_model': EMBEDDING_MODEL,
    'top_k': FAISS_TOP_K,
}

metrics_file = DRIVE_OUTPUTS + 'flat_rag_metrics.json'
with open(metrics_file, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\nMetrics saved to {metrics_file}')
print('\n✅ Flat RAG baseline complete!')